# Notebook 02: Gap and Decision v0.2
## CanonicalPacket → DecisionResult

This notebook imports the v0.2 decision engine from `src/intake/decision.py` and demonstrates it on real NB01 packet output.

**Key Design:**
1. Ambiguity-aware blocker evaluation (blocking vs advisory)
2. Urgency-aware soft blocker suppression
3. Budget feasibility enforcement
4. Operating mode routing (emergency, audit, coordinator, etc.)
5. Contradiction classification with precedence
6. Risk flag generation from fact combinations
7. Hypotheses never fill hard blockers
8. Traveler-safe impossible when blocking ambiguity exists

## Import the v0.2 Decision Engine

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../src"))

from intake.packet_models import SourceEnvelope
from intake.extractors import ExtractionPipeline
from intake.decision import run_gap_and_decision, DecisionResult

print("NB02 v0.2 decision engine loaded")

## Demo 1: Standard Family Intake

In [ ]:
text = (
    "Family of 4 from Bangalore — 2 adults, 2 kids (ages 8 and 12). "
    "Want to go to Singapore, 5 nights, March 15-20, 2026. "
    "Budget around 3 lakhs total. Kids want kid-friendly activities."
)

envelope = SourceEnvelope.from_freeform(text, source="agency_notes", actor="agent")
packet = ExtractionPipeline().extract([envelope])
result = run_gap_and_decision(packet)

print(f"Decision: {result.decision_state}")
print(f"Hard blockers: {result.hard_blockers}")
print(f"Soft blockers: {result.soft_blockers}")
print(f"Ambiguities: {[(a.field_name, a.severity) for a in result.ambiguities]}")
print(f"Confidence: {result.confidence_score}")
if result.follow_up_questions:
    print(f"\nFollow-up questions:")
    for q in result.follow_up_questions:
        print(f"  [{q['priority']}] {q['question']}")
if result.risk_flags:
    print(f"\nRisk flags:")
    for r in result.risk_flags:
        print(f"  [{r['severity']}] {r['flag']}: {r['message']}")

## Demo 2: Blocking Ambiguity

In [ ]:
text2 = "Family of 4 from Bangalore, want Andaman or Sri Lanka, budget 3L, April 2026."
pkt2 = ExtractionPipeline().extract([SourceEnvelope.from_freeform(text2)])
result2 = run_gap_and_decision(pkt2)

print(f"Decision: {result2.decision_state}")
print(f"Hard blockers: {result2.hard_blockers}")
print(f"Blocking ambiguities: {[(a.field_name, a.ambiguity_type) for a in result2.ambiguities if a.severity == 'blocking']}")

## Demo 3: Emergency Mode

In [ ]:
text3 = "URGENT: Medical emergency! Elderly father chest pain in Singapore. Need help NOW."
pkt3 = ExtractionPipeline().extract([SourceEnvelope.from_freeform(text3)])
result3 = run_gap_and_decision(pkt3)

print(f"Operating mode: {result3.operating_mode}")
print(f"Decision: {result3.decision_state}")
print(f"Soft blockers (should be empty): {result3.soft_blockers}")
print(f"Risk flags: {[(r['flag'], r['severity']) for r in result3.risk_flags]}")

## Demo 4: Budget Feasibility

In [ ]:
text4 = "Family of 6 from Bangalore, want Maldives, budget 1.5L total, April 2026."
pkt4 = ExtractionPipeline().extract([SourceEnvelope.from_freeform(text4)])
result4 = run_gap_and_decision(pkt4)

print(f"Decision: {result4.decision_state}")
print(f"Feasibility: {result4.rationale.get('feasibility')}")
print(f"Hard blockers: {result4.hard_blockers}")
if result4.risk_flags:
    for r in result4.risk_flags:
        print(f"  [{r['severity']}] {r['flag']}: {r['message']}")